# Exercise 1: Data Gathering and Exploratory Analysis

The session collects the apartments for sale on Skopje Flats, a listings portal, and describes them without modelling them. The portal is a stored snapshot in `data/site`, and the notebook serves it from this machine, so every request stays on the local loopback address. The listings are synthetic and describe no real apartment, seller, or agency.

The notebook has two parts:

- Part 1 gathers the listings. It reads one page and turns each listing into a record, follows the pages to the last one, collects a page that a browser has to render and a paged endpoint, and joins the sources into one data frame.
- Part 2 describes that data frame. It classifies the columns, examines what is missing, and summarizes each column and each relationship with statistics and figures.

## Setup

In [ ]:
import json
import re
import threading
from functools import partial
from http.server import SimpleHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path
from urllib.parse import parse_qs, urljoin, urlparse

import matplotlib.pyplot as plt
import pandas as pd
import requests
import seaborn as sns
from bs4 import BeautifulSoup, Tag
from playwright.async_api import async_playwright

DATA = Path("data")

In [ ]:
# The course style.
PALETTE = ["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#E69F00", "#56B4E9", "#1e2a24"]
GROUND, INK, GRID = "#ffffff", "#1e2a24", "#e2e6e4"
plt.rcParams.update(
    {
        "figure.figsize": (8, 4.5),
        "figure.facecolor": GROUND,
        "figure.constrained_layout.use": True,
        "savefig.facecolor": GROUND,
        "savefig.transparent": False,
        "font.family": "sans-serif",
        "font.sans-serif": ["Archivo", "DejaVu Sans"],
        "font.size": 12,
        "text.color": INK,
        "axes.facecolor": GROUND,
        "axes.edgecolor": INK,
        "axes.labelcolor": INK,
        "axes.titleweight": "bold",
        "axes.titlelocation": "left",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "axes.axisbelow": True,
        "axes.prop_cycle": plt.cycler(color=PALETTE),
        "grid.color": GRID,
        "xtick.color": INK,
        "ytick.color": INK,
        "legend.frameon": False,
        "image.cmap": "cividis",
    }
)
sns.set_palette(PALETTE)

In [ ]:
# The statistics endpoint answers /api/listing-stats?page=N from a stored file,
# and every other path is a file under data/site.
STATS = json.loads((DATA / "api" / "listing-stats.json").read_text(encoding="utf-8"))
STATS_PER_PAGE = 50


class Portal(SimpleHTTPRequestHandler):
    extensions_map = {
        **SimpleHTTPRequestHandler.extensions_map,
        ".html": "text/html; charset=utf-8",
    }

    def do_GET(self) -> None:
        url = urlparse(self.path)
        if url.path != "/api/listing-stats":
            return super().do_GET()
        page = int(parse_qs(url.query).get("page", ["1"])[0])
        start = (page - 1) * STATS_PER_PAGE
        body = json.dumps(
            {
                "total": len(STATS),
                "page": page,
                "per_page": STATS_PER_PAGE,
                "results": STATS[start : start + STATS_PER_PAGE],
            }
        ).encode()
        self.send_response(200)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def log_message(self, format: str, *args: object) -> None:
        pass


# Port 0 lets the system choose a free port, so a second running notebook never collides.
server = ThreadingHTTPServer(("127.0.0.1", 0), partial(Portal, directory=DATA / "site"))
threading.Thread(target=server.serve_forever, daemon=True).start()
BASE = f"http://127.0.0.1:{server.server_port}/"
BASE

## Part 1: Gathering the Listings

The portal offers the listings in three forms: pages of static markup, a page that a script fills in after it loads, and an endpoint that returns JSON. Each form needs its own collection technique, and the last section of this part joins what they return.

### 1. Requests and Responses

A request names a resource by its URL, and the response carries a status code, headers, and a body. The `timeout` argument bounds how long the client waits, so a server that never answers raises an error instead of stalling the notebook.

The status code is read first, because a body that arrives with an error code is an error page rather than the data. A code in the 200s means success, a code in the 400s means the request was at fault, such as a request for a page that does not exist, and a code in the 500s means the server failed.

In [ ]:
response = requests.get(urljoin(BASE, "listings/page-1.html"), timeout=10)
response.status_code

`raise_for_status()` turns a code in the 400s or the 500s into an exception, so the code after it only ever sees a successful response. The headers describe the body, and `Content-Type` names its format and its character encoding. `response.content` holds the body as bytes, and `response.text` decodes it into a string.

In [ ]:
response.raise_for_status()
response.headers["Content-Type"], len(response.content)

In [ ]:
print(response.text)

A page that does not exist still returns a response. Its status code says so, and `raise_for_status()` turns that code into an exception.

In [ ]:
missing = requests.get(urljoin(BASE, "listings/page-99.html"), timeout=10)
print(missing.status_code, missing.ok)
try:
    missing.raise_for_status()
except requests.HTTPError as error:
    print(error)

### 2. Selectors

Beautiful Soup parses the markup into a tree of elements, and a CSS selector picks elements out of that tree. `select()` returns every match, and `select_one()` returns the first match.

Four kinds of selector cover most pages:

- a tag selector, such as `h2`, matches by element name,
- a class selector, such as `.price`, matches by the `class` attribute,
- an identifier selector, such as `#listing-2563`, matches by the `id` attribute, which is unique within a page, and
- a descendant selector, such as `ul.features li`, matches the elements on the right that lie anywhere inside an element on the left.

A tag and a class written together, as in `article.listing`, match an element that satisfies both. Each listing on the page is such an element.

In [ ]:
soup = BeautifulSoup(response.text, "lxml")
cards = soup.select("article.listing")
first = cards[0]
len(cards), first["data-id"]

An element carries text and attributes. Indexing an element, as in `first["data-id"]`, reads an attribute, and `get_text()` reads the text inside it.

A selector can match more than intended. The page has more `h2` headings than listings, because its other sections have headings too, and the descendant selector `article.listing h2` keeps only the headings inside a listing.

In [ ]:
{
    "h2": len(soup.select("h2")),
    "article.listing h2": len(soup.select("article.listing h2")),
    "#listing-" + first["data-id"]: len(soup.select(f"#listing-{first['data-id']}")),
    "ul.features li": len(soup.select("ul.features li")),
}

Each listing shows up to three features, which are the rooms, the condition, and the elevator, so twelve listings could hold 36 of them. Counting each feature class shows which one is absent from some listing.

In [ ]:
{
    selector: len(soup.select(selector))
    for selector in [".rooms", ".condition", ".elevator", ".balcony"]
}

An empty match is a normal outcome. `select()` returns an empty list and `select_one()` returns `None`, so the code checks for `None` before it reads an element that may be absent.

In [ ]:
first.select_one(".balcony") is None

### 3. One Record per Element

Each listing becomes one record, and the text and the attributes of its elements become the fields. The fields are of two kinds:

- a required field, such as the price, is present in every listing, so its absence means the page has changed and the parser raises an error, and
- an optional field, such as the condition, may be absent, and an absent element becomes a missing value.

The price and the summary hold their numbers inside a sentence.

In [ ]:
print(first.select_one(".price").get_text())
print(first.select_one(".summary").get_text())

A regular expression reads those numbers out. The patterns below use these constructs:

- `\d` matches one digit, and `\d{4}` matches exactly four,
- `[\d ]*` matches any run of digits and spaces, because the price uses a space to separate thousands,
- `[\d.]+` matches one or more digits and points, which covers an area with a decimal part,
- `(EUR|MKD)` matches either currency, and
- each pair of parentheses captures the text it matches, and `groups()` returns the captures in order.

`search()` returns `None` when the pattern does not match, so the helper `captures` checks for it just as `required_text` checks for an absent element.

In [ ]:
PRICE_REGEX = re.compile(r"(\d[\d ]*) (EUR|MKD)")
SUMMARY_REGEX = re.compile(
    r"([\d.]+) m² apartment on floor (\d+) of (\d+), built in (\d{4})"
)


def required_text(card: Tag, selector: str) -> str:
    element = card.select_one(selector)
    if element is None:
        raise ValueError(f"Listing {card['data-id']} has no {selector} element.")
    return element.get_text(strip=True)


def optional_text(card: Tag, selector: str) -> str | None:
    element = card.select_one(selector)
    return element.get_text(strip=True) if element else None


def captures(pattern: re.Pattern[str], text: str) -> tuple[str, ...]:
    match = pattern.search(text)
    if match is None:
        raise ValueError(f"{text!r} does not match {pattern.pattern!r}.")
    return match.groups()


def parse_listing(card: Tag) -> dict[str, object]:
    amount, currency = captures(PRICE_REGEX, required_text(card, ".price"))
    area, floor, floors, year = captures(SUMMARY_REGEX, required_text(card, ".summary"))
    elevator = optional_text(card, ".elevator")
    return {
        "listing_id": int(card["data-id"]),
        "municipality": required_text(card, ".title").split(" in ")[-1],
        "price": int(amount.replace(" ", "")),
        "currency": currency,
        "area": float(area),
        "rooms": int(required_text(card, ".rooms").split()[0]),
        "floor": int(floor),
        "floors": int(floors),
        "year_built": int(year),
        "condition": optional_text(card, ".condition"),
        "elevator": None if elevator is None else elevator.endswith("yes"),
    }


parse_listing(first)

The other fields are read with plain string methods:

- the municipality is the text after the last " in " of a title such as "3-room apartment in Гази Баба",
- the room count is the first word of a feature such as "3 rooms", and
- the elevator is `True` when its feature reads "Elevator: yes", `False` when it reads "Elevator: no", and missing when the feature is absent.

### 4. Pagination

The listings span several pages, and each page except the last links to the next one. The scraper follows that link until a page has none. `urljoin()` resolves the link, which is relative to the current page, into a full URL. Each page adds its records to one list, and the list becomes a data frame at the end.

In [ ]:
records, url, pages = [], urljoin(BASE, "listings/page-1.html"), 0
while url:
    page = requests.get(url, timeout=10)
    page.raise_for_status()
    soup = BeautifulSoup(page.text, "lxml")
    records += [parse_listing(card) for card in soup.select("article.listing")]
    link = soup.select_one("a.next")
    url = urljoin(url, link["href"]) if link else None
    pages += 1

listings = pd.DataFrame(records)
pages, listings.shape

In [ ]:
listings.head()

### 5. A Page That Renders Itself

The market index page builds its table with a script after it loads. The markup that `requests` receives holds only the empty table, because `requests` downloads the markup and runs no script.

In [ ]:
static = BeautifulSoup(
    requests.get(urljoin(BASE, "market-index.html"), timeout=10).text, "lxml"
)
len(static.select("#index tr"))

A browser driven by Playwright runs the script. The table appears some time after the page loads, so the browser waits until a table cell exists before it reads the markup. The notebook already runs an event loop, so it uses the asynchronous interface of Playwright and awaits each step.

The rendered table starts with a header row of `th` cells, and the selector `tr:has(td)` keeps only the rows that contain data cells.

In [ ]:
async with async_playwright() as playwright:
    browser = await playwright.chromium.launch()
    tab = await browser.new_page()
    await tab.goto(urljoin(BASE, "market-index.html"))
    await tab.wait_for_selector("#index td")
    rendered = await tab.content()
    await browser.close()

rows = BeautifulSoup(rendered, "lxml").select("#index tr:has(td)")
market_index = pd.DataFrame(
    [
        {
            "municipality": row.select_one(".municipality").get_text(),
            "portal_eur_per_m2": int(row.select_one(".eur-per-m2").get_text()),
        }
        for row in rows
    ]
)
market_index

### 6. A Paged Endpoint

The statistics endpoint returns JSON, a fixed number of records per page, and each page reports the total number of records.

In [ ]:
body = requests.get(
    urljoin(BASE, "api/listing-stats"), params={"page": 1}, timeout=10
).json()
{key: body[key] for key in ("total", "page", "per_page")}, body["results"][0]

The client asks for the next page until the records it holds account for the total. It also stops on an empty page, so that an endpoint that ends before its reported total cannot trap the client in the loop.

In [ ]:
collected, page_number = [], 1
while True:
    body = requests.get(
        urljoin(BASE, "api/listing-stats"), params={"page": page_number}, timeout=10
    ).json()
    collected += body["results"]
    if len(collected) >= body["total"] or not body["results"]:
        break
    page_number += 1

stats = pd.DataFrame(collected)
len(stats), body["total"], len(listings)

The endpoint holds one record more than the pages hold listings. The join in the next section finds the reason.

### 7. Joining Sources

The listings and the statistics share `listing_id`. Each listing should have exactly one statistics record, so the join is declared as one row against one row, and pandas raises an error when either side repeats a key.

In [ ]:
try:
    listings.merge(stats, on="listing_id", validate="one_to_one")
except pd.errors.MergeError as error:
    print(error)

In [ ]:
stats[stats.duplicated("listing_id", keep=False)]

The two rows are identical, and they sit at positions 149 and 150, which are the last record of page 3 and the first record of page 4. The endpoint served the same record on both sides of a page boundary. `drop_duplicates()` removes only rows that agree in every column, so a repeated key whose rows disagreed would survive it and the join would still fail, which is the right outcome for records that contradict each other.

The join keeps every listing, because `how="left"` keeps each row of the left frame whether or not it finds a match. The indicator column records where each row found its match, and it confirms that every listing found its statistics.

In [ ]:
stats = stats.drop_duplicates()
df = listings.merge(
    stats, on="listing_id", how="left", validate="one_to_one", indicator=True
)
df["_merge"].value_counts()

In [ ]:
df = df.drop(columns="_merge")
df.shape

In [ ]:
df

## Part 2: Describing the Listings

The data frame now holds one row per listing. This part describes it without modelling it: it classifies the columns, examines what is missing, and summarizes each column and each relationship.

### 8. Anscombe's Quartet

Before the listings are described, four small samples show why every measured quantity is also drawn. The four samples share their means, their standard deviations, and their correlation.

In [ ]:
anscombe = pd.read_csv(DATA / "anscombe.csv")
groups = anscombe.groupby("dataset")
summary = groups[["x", "y"]].agg(["mean", "std"])
summary["r"] = groups.apply(lambda g: g["x"].corr(g["y"]))
summary.round(2)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 4), sharex=True, sharey=True)
for ax, (name, group) in zip(axes, groups):
    sns.regplot(
        group,
        x="x",
        y="y",
        ci=None,
        ax=ax,
        scatter_kws={"color": INK, "s": 20},
        line_kws={"color": PALETTE[2]},
    )
    ax.set_title(f"Dataset {name}")
plt.show()

The same summary describes a straight line with noise, a curve, a line with one outlier, and a vertical stack with one far point. The summary alone hides all of that.

### 9. Variable Types

The type of a column decides which summaries and which figures suit it. Each column of the listings is one of these types:

- **quantitative and continuous**, such as `price` and `area`, whose values can fall anywhere in a range,
- **quantitative and discrete**, such as `rooms`, `floor`, `floors`, `year_built`, `days_listed`, and `views`, whose values are whole numbers,
- **categorical and nominal**, such as `municipality`, `currency`, `seller`, `agency`, and the two-valued `elevator`, whose labels have no order, or
- **categorical and ordinal**, such as `condition`, whose labels have an order but no distance between them.

`listing_id` is an identifier. It is stored as a number, but it names a record rather than measuring anything, so its mean or its correlation means nothing. `price` mixes two currencies, which Section 13 handles.

A label column is stored as a category. An ordered category keeps the order of an ordinal variable, so it sorts and compares by meaning rather than alphabetically. The nullable `boolean` type keeps a missing elevator as a missing value, where a plain Boolean column would have no way to store it.

In [ ]:
for column in ["municipality", "currency", "seller", "agency"]:
    df[column] = df[column].astype("category")
df["condition"] = pd.Categorical(
    df["condition"],
    categories=["needs renovation", "good", "renovated", "new build"],
    ordered=True,
)
df["elevator"] = df["elevator"].astype("boolean")
df.dtypes

An ordered category supports comparisons, so the next cell counts the listings that are renovated or newly built. A missing condition compares as false, so a listing without a condition is not counted.

In [ ]:
(df["condition"] >= "renovated").sum()

### 10. The Shape of the Missingness

Counting the missing values says how many there are. Drawing them and crossing them against the other columns shows whether they follow a pattern, and a pattern suggests a reason that the data alone cannot confirm.

In [ ]:
df.isna().sum()[lambda counts: counts > 0]

The map below draws one column per listing, sorted from the oldest building on the left to the newest on the right, and marks each missing value.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 2.6))
missing_map = df.sort_values("year_built")[["agency", "condition", "elevator"]].isna().T
sns.heatmap(
    missing_map, cbar=False, cmap=["#f1f3f2", PALETTE[1]], xticklabels=False, ax=ax
)
ax.set(
    title="Missing values, listings sorted by year built",
    xlabel="Listing, oldest on the left",
)
ax.tick_params(axis="y", rotation=0)
plt.show()

The `elevator` gaps gather among the oldest buildings, while the `agency` and `condition` gaps spread across all ages. Crossing each column with a candidate cause tests what the map suggests.

In [ ]:
pd.crosstab(df["seller"], df["agency"].isna().rename("agency missing"))

In [ ]:
old_and_low = ((df["year_built"] < 1980) & (df["floors"] <= 4)).rename(
    "built before 1980, four floors or fewer"
)
by_building = (
    df["elevator"]
    .isna()
    .groupby(old_and_low)
    .agg(missing="sum", listings="size", share="mean")
)
by_building.round(2)

In [ ]:
by_seller = (
    df["condition"]
    .isna()
    .groupby(df["seller"], observed=True)
    .agg(missing="sum", listings="size", share="mean")
)
by_seller.round(2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2), sharey=True)
by_building["share"].plot.bar(ax=axes[0], rot=0, title="Elevator missing")
by_seller["share"].plot.bar(ax=axes[1], rot=0, title="Condition missing")
axes[0].set_xlabel("Built before 1980, four floors or fewer")
axes[0].set_ylabel("Share of listings")
plt.show()

Each column with gaps follows its own pattern:

- `agency` is missing for every private seller and for no agency listing, because the field does not apply to a private seller,
- `elevator` is missing for about half of the listings in buildings from before 1980 with four floors or fewer, and for very few of the others, and
- `condition` is missing far more often for private sellers than for agencies.

A missing value whose presence depends on another column is not a random gap. Exercise 3 presents the treatment of missing values, and the right treatment depends on which kind of gap it is.

### 11. Central Tendency and Spread

The mean and the median locate the centre of a column. The mean uses every value, so a few extreme values pull it, while the median is the middle value and ignores how far the extremes lie.

The range, the variance, the standard deviation, and the interquartile range measure the spread:

- the range is the distance from the smallest value to the largest,
- the variance is the mean squared distance from the mean, in squared units, and pandas divides by one less than the count, which gives the sample variance,
- the standard deviation is the square root of the variance, back in the units of the column, and
- the interquartile range is the distance from the first quartile to the third, which holds the middle half of the values.

In [ ]:
def centre_and_spread(values: pd.Series) -> pd.Series:
    q1, q3 = values.quantile([0.25, 0.75])
    return pd.Series(
        {
            "mean": values.mean(),
            "median": values.median(),
            "range": values.max() - values.min(),
            "variance": values.var(),
            "std": values.std(),
            "Q1": q1,
            "Q3": q3,
            "IQR": q3 - q1,
        }
    )


pd.DataFrame(
    {"area": centre_and_spread(df["area"]), "views": centre_and_spread(df["views"])}
).round(1)

For `area` the mean and the median nearly agree. For `views` the mean lies far above the median and the standard deviation exceeds the mean, which points to a long right tail: a few listings with very many views pull the mean and the standard deviation up. For such a column the median and the interquartile range describe a typical listing better.

The mode is the most frequent value. It applies to a discrete column and to a label column alike, where the mean and the median do not apply to labels. `mode()` returns a series, because several values can tie for the most frequent, so the cell takes the first one.

In [ ]:
df["rooms"].mode().iloc[0], df["municipality"].mode().iloc[0]

### 12. Histograms and Box Plots

A histogram divides the range of a column into bins and counts the values in each bin, so it shows the shape of a distribution. Its bin count changes what it shows. Too few bins hide the shape, and too many turn it into noise.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.2), sharey=False)
for ax, bins in zip(axes, [5, 20, 80]):
    ax.hist(df["views"], bins=bins)
    ax.set(title=f"{bins} bins", xlabel="Views")
axes[0].set_ylabel("Listings")
plt.show()

A box plot draws the same column more compactly. The box spans the interquartile range, and the line inside it is the median. The whiskers reach the furthest values that lie within 1.5 interquartile ranges of the box, and each value beyond them is drawn as a separate point.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 2.2))
sns.boxplot(df, x="views", ax=ax, color=PALETTE[5])
ax.set_title("Views")
plt.show()

The skewness coefficient summarizes the asymmetry in one number. It is close to zero for a symmetric distribution, positive when the long tail lies on the right, and negative when it lies on the left. A value beyond 1 in either direction marks a strongly skewed column.

In [ ]:
df[["area", "views"]].skew().round(2)

`area` is nearly symmetric, and `views` is strongly skewed to the right, as both figures show.

### 13. A Column That Mixes Units

`price` holds amounts in two currencies, EUR and MKD, and a flat priced in MKD carries a number about sixty times larger than the same flat priced in EUR. A summary of the pooled column mixes two units, so the column is compared within each currency before it is converted.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
sns.histplot(df, x="price", bins=30, ax=axes[0])
axes[0].set_title("Price, both currencies")
sns.boxplot(df, x="currency", y="price", ax=axes[1], log_scale=True)
axes[1].set_title("Price by currency")
plt.show()

In [ ]:
pooled = pd.DataFrame(
    {
        "count": [len(df)],
        "median": [df["price"].median()],
        "skew": [df["price"].skew()],
    },
    index=pd.Index(["pooled"], name="currency"),
)
by_currency = df.groupby("currency", observed=True)["price"].agg(
    ["count", "median", "skew"]
)
pd.concat([pooled, by_currency]).round(2)

Pooled, the column is strongly skewed, because the MKD prices form a second cluster far to the right of the EUR prices. Within each currency the skew is smaller, so much of the pooled skew comes from mixing the units rather than from the prices themselves.

Converting every price to EUR at a fixed rate makes the column comparable, and the price per square metre makes flats of different sizes comparable.

In [ ]:
MKD_PER_EUR = 61.5
df["price_eur"] = df["price"].where(df["currency"] == "EUR", df["price"] / MKD_PER_EUR)
df["eur_per_m2"] = df["price_eur"] / df["area"]
df[["price_eur", "eur_per_m2"]].agg(["median", "skew"]).round(2)

After the conversion, both columns are only moderately skewed.

### 14. Counts of a Category

A categorical column is described by the count and the share of each label, and drawn as a bar plot. A nominal variable is sorted by count, so the bars read from the most common label to the least common, and an ordinal one keeps its own order.

In [ ]:
counts = df["municipality"].value_counts()
pd.DataFrame({"count": counts, "share": (counts / counts.sum()).round(3)})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
sns.countplot(df, y="municipality", order=counts.index, ax=axes[0], color=PALETTE[0])
axes[0].set_title("Listings by municipality")
sns.countplot(df, x="condition", ax=axes[1], color=PALETTE[2])
axes[1].set_title("Listings by condition")
plt.show()

The condition plot counts only the listings that have a condition, so its bars sum to fewer listings than the data frame holds.

In [ ]:
df["condition"].notna().sum(), len(df)

### 15. Relationships and Correlation

A scatter plot shows how two quantitative columns move together, with one point per listing.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
sns.scatterplot(df, x="area", y="price_eur", hue="currency", ax=axes[0])
axes[0].set_title("Price against area")
sns.scatterplot(df, x="days_listed", y="views", ax=axes[1], color=PALETTE[3])
axes[1].set_title("Views against days listed")
plt.show()

A correlation coefficient summarizes such a relationship in one number between −1 and 1:

- the Pearson coefficient measures how close the points lie to a straight line, and
- the Spearman coefficient is the Pearson coefficient of the ranks, so it measures how close the relationship is to monotonic, meaning that one column rises whenever the other does, whatever the shape.

A rank does not grow with the distance of an extreme value, so the Spearman coefficient is less sensitive to a long tail.

In [ ]:
pairs = [("area", "price_eur"), ("days_listed", "views"), ("eur_per_m2", "views")]
pd.DataFrame(
    {
        "pearson": [df[a].corr(df[b]) for a, b in pairs],
        "spearman": [df[a].corr(df[b], method="spearman") for a, b in pairs],
    },
    index=[f"{a} and {b}" for a, b in pairs],
).round(2)

For `area` and `price_eur` the two coefficients agree. For `days_listed` and `views` the Spearman coefficient is the larger one, because the long tail of `views` weakens the straight-line fit more than it weakens the order.

The heatmap below shows the Spearman coefficient of every pair of numeric columns. `listing_id` is left out, because it is an identifier.

In [ ]:
numeric = [
    "price_eur",
    "area",
    "eur_per_m2",
    "rooms",
    "floor",
    "floors",
    "year_built",
    "days_listed",
    "views",
]
fig, ax = plt.subplots(figsize=(8, 6.4))
sns.heatmap(
    df[numeric].corr(method="spearman"),
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    vmin=-1,
    vmax=1,
    square=True,
    ax=ax,
)
ax.set_title("Spearman correlation")
plt.show()

Correlation is not causation. A correlation says that two columns move together, but not which one moves the other, or whether a third factor moves both:

- views and days listed correlate, and here the mechanism is known, because views accumulate while a listing stays online, and
- views and the price per square metre correlate negatively, which fits the explanation that buyers look more at cheaper flats, but the data alone cannot confirm that explanation or rule out another one.

### 16. Grouping and Aggregation

Grouping splits the rows by a label, aggregation reduces each group to a few numbers, and sorting and pivoting arrange the result to answer a question. Here the question is whether the market index of the portal, which it describes as the mean price per square metre with MKD prices converted to EUR, agrees with the listings.

In [ ]:
by_municipality = (
    df.groupby("municipality", observed=True)
    .agg(
        listings=("listing_id", "size"),
        mean_eur_per_m2=("eur_per_m2", "mean"),
        median_eur_per_m2=("eur_per_m2", "median"),
    )
    .round(0)
    .reset_index()
    .astype({"municipality": str})
    .merge(market_index, on="municipality", validate="one_to_one")
    .sort_values("mean_eur_per_m2", ascending=False)
)
by_municipality

The mean computed from the listings reproduces the index of the portal in every municipality. The median lies below the mean in every municipality, because a few expensive flats pull the mean up.

A pivot table crosses two labels. Each cell below holds the number of listings for one municipality and one condition.

In [ ]:
cell_counts = df.pivot_table(
    index="municipality",
    columns="condition",
    values="listing_id",
    aggfunc="size",
    observed=True,
    fill_value=0,
).loc[by_municipality["municipality"]]
cell_counts

A median built from one or two listings is as noisy as its count suggests, so the heatmap of the median price per square metre leaves blank every cell with fewer than three listings.

In [ ]:
MIN_LISTINGS = 3
pivot = df.pivot_table(
    index="municipality",
    columns="condition",
    values="eur_per_m2",
    aggfunc="median",
    observed=True,
).loc[cell_counts.index, cell_counts.columns]
fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(
    pivot, mask=cell_counts < MIN_LISTINGS, annot=True, fmt=".0f", cmap="cividis", ax=ax
)
ax.set_title(
    f"Median EUR per m² by municipality and condition, {MIN_LISTINGS} or more listings"
)
ax.grid(False)
plt.show()

In [ ]:
server.shutdown()